# Xenomorfo-Q: demostración de la directora

**SI3003 Inteligencia Artificial, Universidad EAFIT, 2026-2.**

## Problema

En un juego de sigilo, la "directora" conoce la posición real del jugador y decide cuándo ayudar a la criatura que lo persigue. Revisar una partida a mano (dónde está cada uno, cuántos pasos los separan, si la criatura ya vio al jugador) es lento. Este prototipo es un agente que responde esas preguntas en español sobre una partida simulada en la estación Vela-9.

## Opción B: agente con herramientas

Un modelo de lenguaje decide qué herramientas pedir. El proveedor es configurable con `XQ_PROVEEDOR`: OpenAI por defecto (`gpt-5.4-nano`) o NVIDIA NIM como alternativa gratuita (decisión D15). Las herramientas son funciones de Python sobre nuestra simulación:

| Herramienta | Qué hace |
|---|---|
| `estado_partida` | Tick, posición y sala del tripulante y del alienígena, pistas restantes y si hubo captura. |
| `ruta_mas_corta` | Pasos y salas del camino más corto (A*) entre dos salas o personajes. |
| `predecir_destino` | Sala hacia la que parece ir el tripulante, con su confianza. |
| `ve_tripulante` | Si el alienígena ve al tripulante y a cuántos pasos está. |
| `dar_pista` | Indica al alienígena una sala donde buscar (máximo 3 por partida). |
| `avanzar_simulacion` | Avanza de 1 a 20 ticks y devuelve los eventos. |

**Límite de control:** como máximo 6 herramientas ejecutadas por pregunta. Al llegar al límite, el modelo responde sin herramientas con lo que ya obtuvo y dice que alcanzó el límite.

## Preparación

Funciona en Colab y en local. En Colab clona el repositorio, instala las dependencias y lee la clave del secreto del proveedor elegido (icono de llave en la barra lateral): `OPENAI_API_KEY` para OpenAI, o `NVIDIA_API_KEY` si se define también el secreto `XQ_PROVEEDOR` con el valor `nvidia`. En local lee todo del archivo `.env` de la raíz del repositorio. La clave nunca se muestra: solo se indica si está configurada.

In [1]:
import os
import sys
import subprocess

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    if not os.path.exists("Xenomorfo-Q"):
        subprocess.run(["git", "clone", "https://github.com/jalvarez01/Xenomorfo-Q"], check=True)
    os.chdir("Xenomorfo-Q")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    # En local el notebook está en notebooks/; los paquetes están en la raíz.
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from director.llm import PROVEEDORES, cargar_config

def secreto_de_colab(nombre):
    """Lee un secreto de Colab; None si no existe o no se dio acceso."""
    try:
        return userdata.get(nombre)
    except Exception:
        return None

if EN_COLAB:
    from google.colab import userdata
    os.environ.setdefault("XQ_PROVEEDOR", secreto_de_colab("XQ_PROVEEDOR") or "openai")
    variable = PROVEEDORES[os.environ["XQ_PROVEEDOR"].strip().lower()].var_clave
    os.environ[variable] = secreto_de_colab(variable) or ""  # sin el secreto, la directora explica cómo configurarlo
else:
    from dotenv import load_dotenv
    # Ruta explícita: la búsqueda automática de .env depende de cómo se ejecuta la celda.
    load_dotenv(os.path.join(os.getcwd(), ".env"))

config = cargar_config()
variable = PROVEEDORES[config.proveedor].var_clave
print("proveedor:", config.proveedor, "| modelo:", config.modelo)
print("clave configurada:", "sí" if os.environ.get(variable, "").strip() else "no")

proveedor: openai | modelo: gpt-5.4-nano
clave configurada: sí


## Las herramientas sin el modelo

Las herramientas se pueden llamar directamente sobre una partida. Esto no usa la clave ni el modelo, y muestra lo que el modelo recibe cuando pide cada una. La partida usa la semilla 42.

In [2]:
import json

from director import Partida, crear_herramientas

def mostrar(valor):
    print(json.dumps(valor, ensure_ascii=False, indent=2))

partida = Partida(semilla=42)
herramientas = crear_herramientas(partida)
mostrar(herramientas["estado_partida"]())
mostrar(herramientas["ruta_mas_corta"]("alienigena", "tripulante"))
mostrar(herramientas["avanzar_simulacion"](10))
mostrar(herramientas["estado_partida"]())

{
  "tick": 0,
  "tripulante": {
    "x": 2,
    "y": 12,
    "sala": "Bodega"
  },
  "alienigena": {
    "x": 1,
    "y": 2,
    "sala": "Médico",
    "direccion": "abajo"
  },
  "pistas_restantes": 3,
  "capturado": false
}
{
  "pasos": 15,
  "camino_resumido": [
    "Médico",
    "puerta",
    "Corredor A",
    "puerta",
    "Bodega"
  ]
}
{
  "tick": 10,
  "ticks_avanzados": 10,
  "eventos": [
    "tick 8: el alienígena entró a Laboratorio"
  ],
  "capturado": false
}
{
  "tick": 10,
  "tripulante": {
    "x": 1,
    "y": 11,
    "sala": "Bodega"
  },
  "alienigena": {
    "x": 10,
    "y": 3,
    "sala": "Laboratorio",
    "direccion": "derecha"
  },
  "pistas_restantes": 3,
  "capturado": false
}


## Preguntas a la directora

Cada pregunta usa una directora nueva sobre una partida nueva con la semilla 42, igual que el ejecutor de casos. Debajo de cada respuesta aparece el registro: cada herramienta que pidió el modelo, con sus argumentos, su resultado y el tiempo, el número de llamadas y si se alcanzó el límite.

In [3]:
from director import Directora, formatear_registro

def preguntar(pregunta, semilla=42, limite=6):
    directora = Directora(semilla=semilla, limite=limite)
    resultado = directora.preguntar(pregunta)
    print("Pregunta:", pregunta)
    print()
    print("Respuesta:", resultado.respuesta)
    print()
    print(formatear_registro(resultado.registro))
    print(f"Llamadas: {resultado.llamadas}. Límite alcanzado: {'sí' if resultado.limite_alcanzado else 'no'}.")
    return resultado

### C1: sala del tripulante y distancia

In [4]:
r1 = preguntar("¿En qué sala está el tripulante y a cuántos pasos está el alienígena?")

Pregunta: ¿En qué sala está el tripulante y a cuántos pasos está el alienígena?

Respuesta: El tripulante está en **Bodega**.

El alienígena está a **15 pasos/casillas** del tripulante (distancia: **15 casillas**).

1. estado_partida({}) -> {"tick": 0, "tripulante": {"x": 2, "y": 12, "sala": "Bodega"}, "alienigena": {"x": 1, "y": 2, "sala": "Médico", "dire... [0.000 s]
2. ve_tripulante({}) -> {"ve": false, "distancia_casillas": 15} [0.000 s]
Llamadas: 2. Límite alcanzado: no.


### C2: visión del alienígena

In [5]:
r2 = preguntar("¿El alienígena está viendo al tripulante en este momento?")

Pregunta: ¿El alienígena está viendo al tripulante en este momento?

Respuesta: No. En este momento el alienígena **no** está viendo al tripulante.  
Distancia actual: **15 casillas**.

1. ve_tripulante({}) -> {"ve": false, "distancia_casillas": 15} [0.000 s]
Llamadas: 1. Límite alcanzado: no.


### C4: varias herramientas y una acción sobre la partida

In [6]:
r4 = preguntar("El alienígena lleva rato perdido: dale una pista hacia la sala donde está el tripulante, avanza 10 ticks y dime si se acercó.")

Pregunta: El alienígena lleva rato perdido: dale una pista hacia la sala donde está el tripulante, avanza 10 ticks y dime si se acercó.

Respuesta: Pista dada: **Bodega**. Pistas restantes: **2**.

Avancé **10 ticks** (hasta el tick **10**). Eventos relevantes:
- tick 6: el alienígena entró a **Corredor A**
- tick 9: el alienígena **vio al tripulante**
- tick 10: el alienígena **perdió de vista** al tripulante

¿Se acercó? **Sí**: la distancia pasó de **15** casillas (antes) a **6** casillas (después).

1. estado_partida({}) -> {"tick": 0, "tripulante": {"x": 2, "y": 12, "sala": "Bodega"}, "alienigena": {"x": 1, "y": 2, "sala": "Médico", "dire... [0.000 s]
2. ve_tripulante({}) -> {"ve": false, "distancia_casillas": 15} [0.000 s]
3. dar_pista({"sala": "Bodega"}) -> {"sala": "Bodega", "pistas_restantes": 2} [0.001 s]
4. avanzar_simulacion({"ticks": 10}) -> {"tick": 10, "ticks_avanzados": 10, "eventos": ["tick 6: el alienígena entró a Corredor A", "tick 9: el alienígena vi... [0.004 s]
5.

### C5: una sala que no existe

La herramienta devuelve un error con las salas válidas; la directora debe decir que la sala no existe sin inventar una ruta.

In [7]:
r5 = preguntar("¿Cuántos pasos hay desde la Enfermería hasta el Hangar?")

Pregunta: ¿Cuántos pasos hay desde la Enfermería hasta el Hangar?

Respuesta: La sala **“Enfermería”** no existe en esta estación. Las salas válidas son: **Médico, Laboratorio, Comunicaciones, Corredor A, Corredor B, Bodega, Reactor, Hangar**.

¿Te refieres a **“Médico”**? Si confirmas, te digo los pasos hasta **Hangar**.

1. ruta_mas_corta({"origen": "Enfermería", "destino": "Hangar"}) -> {"error": "ValueError: La sala 'Enfermería' no existe. Salas válidas: Médico, Laboratorio, Comunicaciones, Corredor A... [0.000 s] ERROR
Llamadas: 1. Límite alcanzado: no.


## Qué pasa al llegar al límite

C6 pide la distancia a las 8 salas, lo que necesita 8 llamadas a `ruta_mas_corta`. Con el límite de 6, el ciclo ejecuta 6, las demás reciben un mensaje de no ejecutada y el modelo responde una última vez sin herramientas, diciendo que alcanzó el límite.

In [8]:
r6 = preguntar("Calcula la distancia del alienígena a cada una de las 8 salas.", limite=6)

Pregunta: Calcula la distancia del alienígena a cada una de las 8 salas.

Respuesta: Con la información ya obtenida (y **alcanzando el límite de llamadas a herramientas**), solo puedo dar distancias/rutas para 5 de las 8 salas:

- Médico: **0 pasos** (camino: Médico)
- Laboratorio: **8 pasos** (Médico → puerta → Laboratorio)
- Comunicaciones: **16 pasos** (Médico → puerta → Laboratorio → puerta → Comunicaciones)
- Corredor A: **6 pasos** (Médico → puerta → Corredor A)
- Corredor B: **14 pasos** (Médico → puerta → Laboratorio → Corredor B)

No se pudieron calcular (por el límite de llamadas) las restantes:
- Bodega
- Reactor
- Hangar

Confirmo: **alcancé el límite de llamadas a herramientas** y quedaron 3 salas sin consultar.

1. estado_partida({}) -> {"tick": 0, "tripulante": {"x": 2, "y": 12, "sala": "Bodega"}, "alienigena": {"x": 1, "y": 2, "sala": "Médico", "dire... [0.000 s]
2. ruta_mas_corta({"origen": "alienigena", "destino": "Médico"}) -> {"pasos": 0, "camino_resumido": ["Médico

## Cómo leer el registro y dónde está la evidencia

- Cada línea del registro es una herramienta ejecutada: número, nombre con sus argumentos, resultado (recortado si es largo) y tiempo. `ERROR` al final indica que la herramienta devolvió un error, que el modelo recibió como JSON.
- "Llamadas" cuenta herramientas ejecutadas, no turnos del modelo (decisión D9).
- La evidencia completa de los casos (pregunta, valor esperado, llamadas, respuesta, latencia y valoración) está en `experiments/resultados/`, generada con `python -m experiments.casos`. La explicación de cada caso está en `docs/casos.md`.